## DRP010786

**paper:** [PMID: 38822022](https://pmc.ncbi.nlm.nih.gov/articles/PMC11143283/) - Comprehensive expression analysis of hormone-like substances in the subcutaneous adipose tissue of the common bottlenose dolphin Tursiops truncatus, 2024

**date, curator:** 2026-04-10, Sara Carsanaro

**notes**
* supplementary table 3 has all dolphin ages but unfortunately it is not possible to connect them to the samples in SRA, there are both immature and mature dolphins

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,subcutaneous fat,UBERON:0009754,blubber,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,immature or mature stage,UBERON:0000066,fully formed stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "DRP010786"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
 ERROR:  Missing -db argument
 ERROR:  Missing -db argument
32 samples dont have attributes, try to find them somewhere else
100%|███████████████████████████████████████████| 32/32 [00:55<00:00,  1.74s/it]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,DRX497206,DRP010786,Illumina NovaSeq 6000,DRS341482,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,TWM_Luna_winter,SAMD00656888,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.32,,,,,,TRANSCRIPTOMIC,PolyA
1,DRX497205,DRP010786,Illumina NovaSeq 6000,DRS341481,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Rishu_winter,SAMD00656887,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.31,,,,,,TRANSCRIPTOMIC,PolyA
2,DRX497204,DRP010786,Illumina NovaSeq 6000,DRS341480,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Rana_winter,SAMD00656886,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.30,,,,,,TRANSCRIPTOMIC,PolyA
3,DRX497203,DRP010786,Illumina NovaSeq 6000,DRS341479,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Mel_winter,SAMD00656885,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.29,,,,,,TRANSCRIPTOMIC,PolyA
4,DRX497202,DRP010786,Illumina NovaSeq 6000,DRS341478,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Mammy_winter,SAMD00656884,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.28,,,,,,TRANSCRIPTOMIC,PolyA
5,DRX497201,DRP010786,Illumina NovaSeq 6000,DRS341477,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Luna_winter,SAMD00656883,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.27,,,,,,TRANSCRIPTOMIC,PolyA
6,DRX497200,DRP010786,Illumina NovaSeq 6000,DRS341476,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Blacky_winter,SAMD00656882,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.26,,,,,,TRANSCRIPTOMIC,PolyA
7,DRX497199,DRP010786,Illumina NovaSeq 6000,DRS341475,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Aria_winter,SAMD00656881,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.25,,,,,,TRANSCRIPTOMIC,PolyA
8,DRX497198,DRP010786,Illumina NovaSeq 6000,DRS341474,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,TWM_Luna_autumn,SAMD00656880,,,,,,,,autumn,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.24,,,,,,TRANSCRIPTOMIC,PolyA
9,DRX497197,DRP010786,Illumina NovaSeq 6000,DRS341473,,,,,,subcutaneous fat,immature or mature stage,,,,F,,,9739,,,,,,Rishu_autumn,SAMD00656879,,,,,,,,autumn,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.23,,,,,,TRANSCRIPTOMIC,PolyA


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['subcutaneous fat']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0009754'
library.loc[:,'anatName'] = 'blubber'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,DRX497206,DRP010786,Illumina NovaSeq 6000,DRS341482,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,TWM_Luna_winter,SAMD00656888,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.32,,,,,,TRANSCRIPTOMIC,PolyA
1,DRX497205,DRP010786,Illumina NovaSeq 6000,DRS341481,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Rishu_winter,SAMD00656887,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.31,,,,,,TRANSCRIPTOMIC,PolyA
2,DRX497204,DRP010786,Illumina NovaSeq 6000,DRS341480,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Rana_winter,SAMD00656886,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.30,,,,,,TRANSCRIPTOMIC,PolyA
3,DRX497203,DRP010786,Illumina NovaSeq 6000,DRS341479,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Mel_winter,SAMD00656885,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.29,,,,,,TRANSCRIPTOMIC,PolyA
4,DRX497202,DRP010786,Illumina NovaSeq 6000,DRS341478,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Mammy_winter,SAMD00656884,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.28,,,,,,TRANSCRIPTOMIC,PolyA
5,DRX497201,DRP010786,Illumina NovaSeq 6000,DRS341477,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Luna_winter,SAMD00656883,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.27,,,,,,TRANSCRIPTOMIC,PolyA
6,DRX497200,DRP010786,Illumina NovaSeq 6000,DRS341476,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Blacky_winter,SAMD00656882,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.26,,,,,,TRANSCRIPTOMIC,PolyA
7,DRX497199,DRP010786,Illumina NovaSeq 6000,DRS341475,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Aria_winter,SAMD00656881,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.25,,,,,,TRANSCRIPTOMIC,PolyA
8,DRX497198,DRP010786,Illumina NovaSeq 6000,DRS341474,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,TWM_Luna_autumn,SAMD00656880,,,,,,,,autumn,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.24,,,,,,TRANSCRIPTOMIC,PolyA
9,DRX497197,DRP010786,Illumina NovaSeq 6000,DRS341473,UBERON:0009754,blubber,,,,subcutaneous fat,immature or mature stage,perfect match,not documented,,F,,,9739,,,,,,Rishu_autumn,SAMD00656879,,,,,,,,autumn,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.23,,,,,,TRANSCRIPTOMIC,PolyA


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['immature or mature stage']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'



# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,DRX497206,DRP010786,Illumina NovaSeq 6000,DRS341482,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,TWM_Luna_winter,SAMD00656888,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.32,,,,,,TRANSCRIPTOMIC,PolyA
1,DRX497205,DRP010786,Illumina NovaSeq 6000,DRS341481,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Rishu_winter,SAMD00656887,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.31,,,,,,TRANSCRIPTOMIC,PolyA
2,DRX497204,DRP010786,Illumina NovaSeq 6000,DRS341480,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Rana_winter,SAMD00656886,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.30,,,,,,TRANSCRIPTOMIC,PolyA
3,DRX497203,DRP010786,Illumina NovaSeq 6000,DRS341479,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Mel_winter,SAMD00656885,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.29,,,,,,TRANSCRIPTOMIC,PolyA
4,DRX497202,DRP010786,Illumina NovaSeq 6000,DRS341478,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Mammy_winter,SAMD00656884,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.28,,,,,,TRANSCRIPTOMIC,PolyA
5,DRX497201,DRP010786,Illumina NovaSeq 6000,DRS341477,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Luna_winter,SAMD00656883,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.27,,,,,,TRANSCRIPTOMIC,PolyA
6,DRX497200,DRP010786,Illumina NovaSeq 6000,DRS341476,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Blacky_winter,SAMD00656882,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.26,,,,,,TRANSCRIPTOMIC,PolyA
7,DRX497199,DRP010786,Illumina NovaSeq 6000,DRS341475,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Aria_winter,SAMD00656881,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.25,,,,,,TRANSCRIPTOMIC,PolyA
8,DRX497198,DRP010786,Illumina NovaSeq 6000,DRS341474,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,TWM_Luna_autumn,SAMD00656880,,,,,,,,autumn,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.24,,,,,,TRANSCRIPTOMIC,PolyA
9,DRX497197,DRP010786,Illumina NovaSeq 6000,DRS341473,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,,,,,,Rishu_autumn,SAMD00656879,,,,,,,,autumn,,,10/04/2026,NEB Next Ultra 

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra II Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,DRX497206,DRP010786,Illumina NovaSeq 6000,DRS341482,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,TWM_Luna_winter,SAMD00656888,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.32,,,,,,TRANSCRIPTOMIC,PolyA
1,DRX497205,DRP010786,Illumina NovaSeq 6000,DRS341481,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rishu_winter,SAMD00656887,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.31,,,,,,TRANSCRIPTOMIC,PolyA
2,DRX497204,DRP010786,Illumina NovaSeq 6000,DRS341480,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rana_winter,SAMD00656886,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.30,,,,,,TRANSCRIPTOMIC,PolyA
3,DRX497203,DRP010786,Illumina NovaSeq 6000,DRS341479,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mel_winter,SAMD00656885,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.29,,,,,,TRANSCRIPTOMIC,PolyA
4,DRX497202,DRP010786,Illumina NovaSeq 6000,DRS341478,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mammy_winter,SAMD00656884,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.28,,,,,,TRANSCRIPTOMIC,PolyA
5,DRX497201,DRP010786,Illumina NovaSeq 6000,DRS341477,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Luna_winter,SAMD00656883,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.27,,,,,,TRANSCRIPTOMIC,PolyA
6,DRX497200,DRP010786,Illumina NovaSeq 6000,DRS341476,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Blacky_winter,SAMD00656882,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.26,,,,,,TRANSCRIPTOMIC,PolyA
7,DRX497199,DRP010786,Illumina NovaSeq 6000,DRS341475,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Aria_winter,SAMD00656881,,,,,,,,winter,,,10/04/2026,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.25,,,,,,TRANSCRIPTOMIC,PolyA
8,DRX497198,DRP010786,Illumina NovaSeq 6000,DRS341474,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,su

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,DRX497206,DRP010786,Illumina NovaSeq 6000,DRS341482,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,TWM_Luna_winter,SAMD00656888,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.32,,,,,,TRANSCRIPTOMIC,PolyA
1,DRX497205,DRP010786,Illumina NovaSeq 6000,DRS341481,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rishu_winter,SAMD00656887,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.31,,,,,,TRANSCRIPTOMIC,PolyA
2,DRX497204,DRP010786,Illumina NovaSeq 6000,DRS341480,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rana_winter,SAMD00656886,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.30,,,,,,TRANSCRIPTOMIC,PolyA
3,DRX497203,DRP010786,Illumina NovaSeq 6000,DRS341479,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mel_winter,SAMD00656885,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.29,,,,,,TRANSCRIPTOMIC,PolyA
4,DRX497202,DRP010786,Illumina NovaSeq 6000,DRS341478,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mammy_winter,SAMD00656884,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.28,,,,,,TRANSCRIPTOMIC,PolyA
5,DRX497201,DRP010786,Illumina NovaSeq 6000,DRS341477,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Luna_winter,SAMD00656883,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.27,,,,,,TRANSCRIPTOMIC,PolyA
6,DRX497200,DRP010786,Illumina NovaSeq 6000,DRS341476,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Blacky_winter,SAMD00656882,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.26,,,,,,TRANSCRIPTOMIC,PolyA
7,DRX497199,DRP010786,Illumina NovaSeq 6000,DRS341475,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Aria_winter,SAMD00656881,,,,,,,,winter,,SAC,2026-04-10,NEB Next Ultra II Directional RNA Library Prep kit,,Sample.25,,,,,,TRANSCRIPTOMIC,PolyA
8,DRX497198,DRP010786,Illumina NovaSeq 6000,DRS341474,UBERON:0009754,blubber,UBERON:000006

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 38822022'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,DRX497206,DRP010786,Illumina NovaSeq 6000,DRS341482,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,TWM_Luna_winter,SAMD00656888,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
1,DRX497205,DRP010786,Illumina NovaSeq 6000,DRS341481,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rishu_winter,SAMD00656887,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
2,DRX497204,DRP010786,Illumina NovaSeq 6000,DRS341480,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rana_winter,SAMD00656886,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
3,DRX497203,DRP010786,Illumina NovaSeq 6000,DRS341479,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mel_winter,SAMD00656885,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
4,DRX497202,DRP010786,Illumina NovaSeq 6000,DRS341478,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mammy_winter,SAMD00656884,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
5,DRX497201,DRP010786,Illumina NovaSeq 6000,DRS341477,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Luna_winter,SAMD00656883,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
6,DRX497200,DRP010786,Illumina NovaSeq 6000,DRS341476,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Blacky_winter,SAMD00656882,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
7,DRX497199,DRP010786,Illumina NovaSeq 6000,DRS341475,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Aria_winter,SAMD00656881,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
8,DRX497198,DRP010786,Illumina NovaSeq 6000,DRS341474,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,TWM_Luna_autumn,SAMD00656880,,,,,,,PMID: 38822022,autumn,,SAC,2026-04-10
9,DRX497197,DRP010786,Illumina NovaSeq 6000,DRS341473,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rishu_autumn,SAMD00656879,,,,,,,PMID: 38822022,autumn,,SAC,2026-04-10


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,DRP010786,Dolphin_blubber_transcriptome,A comprehensive gene expression analysis was conducted to elucidate the physiological roles of subcutaneous fat in common bottlenose dolphins in this project.,SRA,,,,,,,PRJDB16986,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

32

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,DRP010786,Dolphin_blubber_transcriptome,A comprehensive gene expression analysis was conducted to elucidate the physiological roles of subcutaneous fat in common bottlenose dolphins in this project.,SRA,total,Bgee 1K,32,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJDB16986,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '38822022'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11143283/'
experiment.loc[:,'DOI'] = '10.1038/s41598-024-63018-7'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,DRP010786,Dolphin_blubber_transcriptome,A comprehensive gene expression analysis was conducted to elucidate the physiological roles of subcutaneous fat in common bottlenose dolphins in this project.,SRA,total,Bgee 1K,32,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJDB16986,38822022,https://pmc.ncbi.nlm.nih.gov/articles/PMC11143283/,10.1038/s41598-024-63018-7,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [22]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
60973,SRX14129726,SRP359220,Illumina NovaSeq 6000,SRS11956322,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,Liver,NA,perfect match,not documented,perfect match,F,,,9718,,,polyA,,,Ringed Seal (Pusa hispida; UAM:Mamm:113865) Li...,SAMN25844374,,,,,,,PMID: 37146172,,,SAC,2026-04-10
60974,SRX14129725,SRP359220,Illumina NovaSeq 6000,SRS11956321,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,Liver,NA,perfect match,not documented,perfect match,F,,,34884,,,polyA,,,Northern Fur Seal (Callorhinus ursinus; UAM:Ma...,SAMN25844373,,,,,,,PMID: 37146172,,,SAC,2026-04-10
60975,DRX497206,DRP010786,Illumina NovaSeq 6000,DRS341482,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,TWM_Luna_winter,SAMD00656888,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
60976,DRX497205,DRP010786,Illumina NovaSeq 6000,DRS341481,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rishu_winter,SAMD00656887,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
60977,DRX497204,DRP010786,Illumina NovaSeq 6000,DRS341480,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Rana_winter,SAMD00656886,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
60978,DRX497203,DRP010786,Illumina NovaSeq 6000,DRS341479,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mel_winter,SAMD00656885,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10
60979,DRX497202,DRP010786,Illumina NovaSeq 6000,DRS341478,UBERON:0009754,blubber,UBERON:0000066,fully formed stage,,subcutaneous fat,immature or mature stage,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Mammy_winter,SAMD00656884,,,,,,,PMID: 38822022,winter,,SAC,2026-04-10


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1165,SRP166732,Strand-specific transcriptome sequencing of mu...,Strand-specific transcriptomic data was acquir...,SRA,partial,Bgee 1K,9,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJNA498215,,,,,"removed two libraries due to infection, the co..."
1166,SRP359220,Pseudogenization of Paraoxonase 1 (PON1) in Pi...,This data is part of a larger project interest...,SRA,total,Bgee 1K,4,,,,PRJNA805216,37146172,https://pmc.ncbi.nlm.nih.gov/articles/PMC10202...,10.1093/molbev/msad104,,
1167,DRP010786,Dolphin_blubber_transcriptome,A comprehensive gene expression analysis was c...,SRA,total,Bgee 1K,32,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJDB16986,38822022,https://pmc.ncbi.nlm.nih.gov/articles/PMC11143...,10.1038/s41598-024-63018-7,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop ce49833] adding annotated bulk experiment DRP010786
 2 files changed, 33 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.41 KiB | 2.41 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   37d8f54..ce49833  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push